<a href="https://colab.research.google.com/github/fernandodeeke/Hydraulics/blob/main/tubulent_2_tank_slide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider, VBox, HBox, Label, HTML
import ipywidgets as widgets

# Enable LaTeX rendering in matplotlib
plt.rcParams.update({
    'text.usetex': False,          # use mathtext (no external LaTeX needed)
    'mathtext.fontset': 'cm',      # Computer Modern – classic LaTeX look
    'font.family': 'serif',
})

# ==============================================================
# TWO-TANK SYSTEM -- INTERACTIVE NOTEBOOK
# Adjust sliders to explore how each physical parameter affects
# the system's transient response and equilibrium heights.
# ==============================================================

# --------------------------------------------------------------
# SLIDER DEFINITIONS
# Each slider description includes: symbol, unit, and meaning.
# Ranges are chosen to stay physically plausible.
# --------------------------------------------------------------

slider_style  = {'description_width': '260px'}
slider_layout = widgets.Layout(width='600px')

def make_slider(desc, val, lo, hi, step, fmt='.4g'):
    return FloatSlider(
        value=val, min=lo, max=hi, step=step,
        description=desc,
        readout_format=fmt,
        style=slider_style, layout=slider_layout,
        continuous_update=True
    )

def header(text):
    """Section header with LaTeX-style math rendered via MathJax in the notebook."""
    style = (
        "font-size:14px; font-weight:bold; color:#2c3e50; "
        "border-bottom:2px solid #ecf0f1; margin-top:15px; "
        "margin-bottom:10px; padding-bottom:5px; width:600px;"
    )
    return HTML(f"<div style='{style}'>{text}</div>")



# ── Initial Conditions ─────────────────────────────────────────
w_h1_0 = make_slider('h₁(0)  Initial level Tank 1 (m)', 0.00, 0.00, 0.40, 0.005, '.3f')
w_h2_0 = make_slider('h₂(0)  Initial level Tank 2 (m)', 0.00, 0.00, 0.40, 0.005, '.3f')

# ── Tank Geometry ──────────────────────────────────────────────
w_A1 = make_slider('A₁  Cross-section Tank 1 (m²)',  0.02, 0.005, 0.10, 0.005, '.4f')
w_A2 = make_slider('A₂  Cross-section Tank 2 (m²)',  0.02, 0.005, 0.10, 0.005, '.4f')

# ── Inflow ─────────────────────────────────────────────────────
# 1 L/min = 1.667 × 10⁻⁵ m³/s
w_q1_lpm = make_slider('q₁  Inflow rate (L/min)', 1.00, 0.10, 5.00, 0.10, '.2f')

# ── Fluid Properties ───────────────────────────────────────────
w_mu  = make_slider('μ   Dynamic viscosity (mPa·s)', 1.00,   0.50, 5.00,   0.10, '.2f')
w_rho = make_slider('ρ   Fluid density (kg/m³)',    1000.0, 800.0, 1200.0, 10.0, '.1f')

# ── Connecting Pipe (Hagen-Poiseuille) ─────────────────────────
w_L = make_slider('L   Pipe length (m)',        0.20, 0.05, 1.00, 0.01, '.3f')
w_r = make_slider('r   Pipe radius (mm)',       1.50, 0.50, 5.00, 0.10, '.2f')

# ── Outlet Valve (Torricelli) ──────────────────────────────────
w_Cd = make_slider('Cᵈ  Discharge coefficient (−)',    0.62, 0.30, 0.90, 0.01,  '.3f')
w_a  = make_slider('a   Valve area (×10⁻⁵ m²)',        1.50, 0.10, 5.00, 0.10,  '.2f')

# ── Simulation Horizon ─────────────────────────────────────────
w_T = make_slider('T   Simulation horizon (s)', 1500.0, 300.0, 3600.0, 60.0, '.0f')


# ==============================================================
# SIMULATION FUNCTION
# All arguments are in SI units.
# Returns the ODE solution and key derived quantities.
# ==============================================================

def run_simulation(h1_0, h2_0, A1, A2, q1, mu, rho, L, r, Cd, a, T):

    g = 9.81  # gravitational acceleration (m/s²)

    # Hagen-Poiseuille hydraulic resistance:
    #   R₁₂ = 8 μ L / (ρ g π r⁴)
    R12 = (8 * mu * L) / (rho * g * np.pi * r**4)

    # Torricelli turbulent valve coefficient:
    #   k₂ = Cᵈ · a · √(2g)
    k2 = Cd * a * np.sqrt(2 * g)

    # Analytical steady-state:
    #   h₂* = (q₁/k₂)²,   h₁* = R₁₂·q₁ + h₂*
    h2_star = (q1 / k2) ** 2
    h1_star = R12 * q1 + h2_star

    # ODE system – mass balances
    def two_tanks(t, h):
        h1 = max(0.0, h[0])
        h2 = max(0.0, h[1])

        q12 = (h1 - h2) / R12        # laminar inter-tank flow
        q2  = k2 * np.sqrt(h2)       # turbulent outlet flow

        dh1_dt = (q1  - q12) / A1
        dh2_dt = (q12 - q2)  / A2
        return [dh1_dt, dh2_dt]

    t_eval = np.linspace(0, T, 1000)
    sol = solve_ivp(
        two_tanks, (0, T), [h1_0, h2_0],
        t_eval=t_eval, method='RK45', max_step=T / 500
    )

    return sol, h1_star, h2_star, R12, k2


# ==============================================================
# PLOT FUNCTION
# Uses matplotlib mathtext for LaTeX-quality axis labels,
# titles, legends, and in-axes annotation boxes.
# ==============================================================

def plot(h1_0, h2_0, A1, A2, q1_lpm, mu_mPas, rho, L, r_mm, Cd, a_1e5, T):

    # Unit conversions
    q1 = q1_lpm  * 1e-3 / 60   # L/min  → m³/s
    mu = mu_mPas * 1e-3         # mPa·s  → Pa·s
    r  = r_mm    * 1e-3         # mm     → m
    a  = a_1e5   * 1e-5         # ×10⁻⁵ → m²

    sol, h1_star, h2_star, R12, k2 = run_simulation(
        h1_0, h2_0, A1, A2, q1, mu, rho, L, r, Cd, a, T
    )

    fig, ax = plt.subplots(figsize=(10, 5.5))

    # ── Transient curves ───────────────────────────────────────
    ax.plot(sol.t, sol.y[0] * 100, color='steelblue',  lw=2.2,
            label=r'$h_1(t)$ — Tank 1')
    ax.plot(sol.t, sol.y[1] * 100, color='darkorange', lw=2.2,
            label=r'$h_2(t)$ — Tank 2')

    # ── Steady-state reference lines ───────────────────────────
    ax.axhline(h1_star * 100, color='steelblue',  lw=1.3, ls='--', alpha=0.65,
               label=r'$h_1^*$ = {:.2f} cm'.format(h1_star * 100))
    ax.axhline(h2_star * 100, color='darkorange', lw=1.3, ls='--', alpha=0.65,
               label=r'$h_2^*$ = {:.2f} cm'.format(h2_star * 100))

    # ── Axis labels & title (LaTeX mathtext) ───────────────────
    ax.set_xlabel(r'Time $t$ (s)', fontsize=13)
    ax.set_ylabel(r'Water level $h$ (cm)', fontsize=13)
    ax.set_title(
        r'Two-Tank System — Transient Response  '
        r'$\left(\dot{h}_1 = \frac{q_1 - q_{12}}{A_1},\quad'
        r'\dot{h}_2 = \frac{q_{12} - q_2}{A_2}\right)$',
        fontsize=12, pad=12
    )

    ax.legend(loc='lower right', fontsize=11,
              framealpha=0.85, edgecolor='#ccc')
    ax.grid(True, ls=':', alpha=0.45)
    ax.set_xlim(0, T)

    # ── Derived-quantity annotation box ────────────────────────
    # Three lines with mathtext equations:
    # Format numbers first to avoid .format() tripping on LaTeX braces
    _R12 = '{:.3f}'.format(R12)
    _k2  = '{:.3e}'.format(k2)
    _h1s = '{:.2f}'.format(h1_star * 100)
    _h2s = '{:.2f}'.format(h2_star * 100)
    line1 = r'$R_{12} = \dfrac{8\,\mu L}{\rho\,g\,\pi\,r^4} = $' + f' {_R12} s/m²'
    line2 = r'$k_2 = C_d\,a\,\sqrt{2g} = $' + f' {_k2} m²·⁵/s'
    line3 = r'$h_1^* = $' + f' {_h1s} cm,   ' + r'$h_2^* = $' + f' {_h2s} cm'

    annotation = '\n'.join([line1, line2, line3])

    ax.text(
        0.015, 0.975, annotation,
        transform=ax.transAxes,
        fontsize=10, va='top', linespacing=2.2,
        bbox=dict(boxstyle='round,pad=0.5', fc='white',
                  ec='#bbb', alpha=0.88)
    )

    plt.tight_layout()
    plt.show()


# ==============================================================
# WIDGET LAYOUT
# Headers use Unicode + MathJax (rendered by the notebook).
# Sliders use Unicode superscripts/subscripts for readability.
# ==============================================================

ui = VBox([
    header('Initial Conditions'),
    w_h1_0, w_h2_0,

    header('Tank Geometry'),
    w_A1, w_A2,

    header('Inflow'),
    w_q1_lpm,

    header('Fluid Properties'),
    w_mu, w_rho,

    header('Connecting Pipe — Hagen-Poiseuille (laminar)'),
    w_L, w_r,

    header('Outlet Valve — Torricelli (turbulent)'),
    w_Cd, w_a,

    header('Simulation'),
    w_T,
])

out = widgets.interactive_output(plot, {
    'h1_0':    w_h1_0,
    'h2_0':    w_h2_0,
    'A1':      w_A1,
    'A2':      w_A2,
    'q1_lpm':  w_q1_lpm,
    'mu_mPas': w_mu,
    'rho':     w_rho,
    'L':       w_L,
    'r_mm':    w_r,
    'Cd':      w_Cd,
    'a_1e5':   w_a,
    'T':       w_T,
})

display(ui, out)


Output()